# CS6493 Topic 4 — Merged Generation Notebook (Base + Instruct, Colab, self-contained lineage)

This notebook merges the two completed generation notebooks into **one replacement notebook** for the project repo:

- the **base-model final notebook** with the improved retrieval-zip workflow and stronger base-generation pipeline
- the **instruction-tuned teammate notebook** with its completed comparison / analysis / debugging outputs

### What is preserved here
- the **improved base-model generation code**
- the **instruction-tuned comparison code path**
- the **saved outputs already present in both source notebooks**
- the **extra debugging / inspection cells** that were added during development

### How conflicts are handled
- duplicated setup sections are kept only once
- the **base final workflow** remains the main backbone
- the preserved teammate comparison section uses **separate variable names** so it does not overwrite the base-final results
- shared helpers are unified so the notebook can still be used as a single Colab notebook

This merged notebook is intended to directly replace the earlier self-contained notebook on GitHub.


## Recommended Colab runtime

In Colab, go to:

**Runtime → Change runtime type → Hardware accelerator → GPU**

A **T4** is usually enough for one **7B base model at a time** with **4-bit quantization**.


In [2]:
# Check your runtime first
!nvidia-smi

Wed Apr  8 09:37:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install dependencies

If you change package versions, restart the runtime after installation.


In [1]:
%pip install -q -U "bitsandbytes>=0.46.1" "transformers>=4.57.0" "accelerate>=1.0.0" datasets pandas tqdm sentencepiece evaluate scikit-learn huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

## Optional: mount Google Drive

Set `USE_GOOGLE_DRIVE = True` if you want:

- model cache to survive Colab restarts
- outputs saved to your Drive
- uploaded zip files copied to Drive


In [3]:
from pathlib import Path
import os

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/cs6493_topic4_generation_base_final")
else:
    BASE_DIR = Path("/content/cs6493_topic4_generation_base_final")

BASE_DIR.mkdir(parents=True, exist_ok=True)

HF_HOME = BASE_DIR / "hf_cache"
OUTPUT_DIR = BASE_DIR / "outputs"
INPUT_DIR = BASE_DIR / "inputs"

HF_HOME.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")

print("BASE_DIR  =", BASE_DIR)
print("INPUT_DIR =", INPUT_DIR)
print("OUTPUT_DIR=", OUTPUT_DIR)


BASE_DIR  = /content/cs6493_topic4_generation_base_final
INPUT_DIR = /content/cs6493_topic4_generation_base_final/inputs
OUTPUT_DIR= /content/cs6493_topic4_generation_base_final/outputs


## Optional: copy uploaded files into `INPUT_DIR`

If you uploaded the zip files through the Colab UI, they often land directly in `/content/`.
This cell copies them into the notebook input directory if found.


In [7]:
import shutil
from pathlib import Path

CANDIDATE_FILES = [
    "/content/RAGProjectRetrive-main.zip",
    "/content/RAGProjectPreprocess-main.zip",
]

for src in CANDIDATE_FILES:
    src_path = Path(src)
    if src_path.exists():
        dst_path = INPUT_DIR / src_path.name
        if not dst_path.exists():
            shutil.copy2(src_path, dst_path)
            print(f"Copied: {src_path} -> {dst_path}")
        else:
            print(f"Already exists: {dst_path}")
    else:
        print(f"Not found (this is okay if you will set the path manually): {src_path}")


Copied: /content/RAGProjectRetrive-main.zip -> /content/cs6493_topic4_generation_base_final/inputs/RAGProjectRetrive-main.zip
Copied: /content/RAGProjectPreprocess-main.zip -> /content/cs6493_topic4_generation_base_final/inputs/RAGProjectPreprocess-main.zip


In [8]:
import gc
import io
import json
import math
import random
import re
import textwrap
import warnings
import zipfile
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


## Configuration

This merged notebook keeps the **base-final retrieval-zip workflow** as the main default, while also exposing the
**instruction-tuned model name** for the preserved comparison section.

### Main default path
- `INPUT_MODE = "retrieval_zip"` to use the team retrieval archive directly
- `RETRIEVAL_VARIANT = "bm25"` by default
- improved **question-aware context compression** is enabled for the base model

### Preserved comparison path
- the teammate comparison block can still run **base vs instruct**
- its result variables are renamed so they do **not overwrite** the base-final outputs

### Supported input modes
- `retrieval_zip`
- `team_jsonl`
- `oracle`


In [9]:
# =========================
# Main configuration
# =========================

SEED = 42
random.seed(SEED)

MODEL_PRESET = "qwen_7b"   # "qwen_7b" or "qwen_3b"

MODEL_OPTIONS = {
    "qwen_7b": "Qwen/Qwen2.5-7B",
    "qwen_3b": "Qwen/Qwen2.5-3B",
}
MODEL_OPTIONS_INSTRUCT = {
    "qwen_7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen_3b": "Qwen/Qwen2.5-3B-Instruct",
}
BASE_MODEL_NAME = MODEL_OPTIONS[MODEL_PRESET]
INSTRUCT_MODEL_NAME = MODEL_OPTIONS_INSTRUCT[MODEL_PRESET]

INPUT_MODE = "retrieval_zip"   # "retrieval_zip", "team_jsonl", or "oracle"

RETRIEVAL_ZIP_PATH = INPUT_DIR / "RAGProjectRetrive-main.zip"
PREPROCESS_ZIP_PATH = INPUT_DIR / "RAGProjectPreprocess-main.zip"  # optional reference only
TEAM_JSONL_PATH = INPUT_DIR / "team_retrieval.jsonl"

RETRIEVAL_VARIANT = "bm25"     # "bm25", "dense", "hybrid_safe"
TOP_K_CONTEXTS = 3
MAX_CONTEXTS_AFTER_HYBRID = 4

# Context cleaning / compression
USE_CONTEXT_COMPRESSION = True
MAX_SENTENCES_PER_CONTEXT = 2
MAX_TOTAL_CONTEXT_CHARS = 3200

# Optional oracle mode continuity
DOMAINS_TO_RUN = ["hotpotqa", "pubmedqa", "financebench"]
SAMPLES_PER_DOMAIN = {
    "hotpotqa": 12,
    "pubmedqa": 12,
    "financebench": 12,
    "natural_questions": 8,
}
USE_STREAMING_FOR_NQ = True

# Model / decoding
USE_4BIT = True
FORCE_CPU = False
MAX_NEW_TOKENS = 96
TEMPERATURE = 0.0
DO_SAMPLE = False
REPETITION_PENALTY = 1.08
NO_REPEAT_NGRAM_SIZE = 4

# Prompt / fallback
USE_FALLBACK_PROMPT = True

OUTPUT_PREFIX = f"{MODEL_PRESET}_base_generation_final"
COMPARISON_OUTPUT_PREFIX = f"{MODEL_PRESET}_topic4_generation_merged_comparison"

print("BASE_MODEL_NAME      =", BASE_MODEL_NAME)
print("INSTRUCT_MODEL_NAME  =", INSTRUCT_MODEL_NAME)
print("INPUT_MODE           =", INPUT_MODE)
print("RETRIEVAL_VARIANT    =", RETRIEVAL_VARIANT)
print("TOP_K_CONTEXTS       =", TOP_K_CONTEXTS)
print("USE_CONTEXT_COMPRESSION =", USE_CONTEXT_COMPRESSION)


BASE_MODEL_NAME      = Qwen/Qwen2.5-7B
INPUT_MODE           = retrieval_zip
RETRIEVAL_VARIANT    = bm25
TOP_K_CONTEXTS       = 3
USE_CONTEXT_COMPRESSION = True


## Optional: Hugging Face login

Most settings here are public. Only use this if needed.


In [10]:
# Optional:
# from huggingface_hub import login
# login()


## Utilities

In [11]:
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

STOPWORDS = {
    "a","an","the","of","in","on","at","to","for","from","by","with","and","or","but","as",
    "is","are","was","were","be","been","being","do","does","did","have","has","had",
    "what","which","who","whom","when","where","why","how","both","same","that","this",
    "these","those","it","its","their","his","her","into","about","than","then","also"
}

def normalize_text(text: Any) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_answer_for_ynm(text: Any) -> str:
    t = normalize_text(text)
    if re.search(r"\byes\b", t):
        return "yes"
    if re.search(r"\bno\b", t):
        return "no"
    if re.search(r"\bmaybe\b", t):
        return "maybe"
    return t

def exact_match_score(pred: Any, gold: Any) -> float:
    pred_n = normalize_answer_for_ynm(pred)
    gold_n = normalize_answer_for_ynm(gold)
    return float(pred_n == gold_n)

def token_f1_score(pred: Any, gold: Any) -> float:
    pred_tokens = normalize_answer_for_ynm(pred).split()
    gold_tokens = normalize_answer_for_ynm(gold).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def tokenize_simple(text: Any) -> List[str]:
    return re.findall(r"[a-z0-9]+", str(text).lower())

def question_keywords(question: str) -> List[str]:
    toks = [t for t in tokenize_simple(question) if t not in STOPWORDS and len(t) > 1]
    return toks

def is_yesno_question(question: str) -> bool:
    q = str(question).strip().lower()
    starters = ("is ", "are ", "was ", "were ", "do ", "does ", "did ", "has ", "have ", "had ", "can ", "could ", "will ", "would ")
    return q.startswith(starters)

def detect_question_type(question: str) -> str:
    q = str(question).strip().lower()
    if is_yesno_question(q):
        return "yes_no"
    if q.startswith("who"):
        return "who"
    if q.startswith("when") or "what year" in q:
        return "when"
    if q.startswith("where"):
        return "where"
    if "how many" in q or "how much" in q:
        return "numeric"
    return "span"

def context_to_text(ctx: Any) -> str:
    if isinstance(ctx, str):
        return ctx.strip()
    if isinstance(ctx, dict):
        title = str(ctx.get("title", "")).strip()
        text = str(ctx.get("text", ctx.get("page_content", ctx.get("evidence_text", ctx.get("chunk_text", ""))))).strip()
        if title and text:
            return f"[{title}] {text}"
        return (title or text).strip()
    return str(ctx).strip()

def contexts_to_list(contexts: Any, top_k: int = 3) -> List[str]:
    out = []
    for c in safe_list(contexts):
        s = context_to_text(c)
        if s:
            out.append(s)
    return out[:top_k]

def join_contexts(contexts: List[str]) -> str:
    pieces = []
    for i, ctx in enumerate(contexts, start=1):
        pieces.append(f"[Document {i}] {ctx}")
    return "\n\n".join(pieces)

def is_abstention(text: Any) -> bool:
    t = str(text).lower()
    triggers = [
        "insufficient evidence",
        "not enough information",
        "cannot answer",
        "can't answer",
        "unknown",
        "not provided in the context",
        "not enough evidence",
        "insufficient information",
    ]
    return any(x in t for x in triggers)

def prediction_in_context(pred: Any, contexts: Any) -> float:
    pred_n = normalize_text(pred)
    ctx_n = normalize_text(" ".join(contexts_to_list(contexts, top_k=999)))
    if not pred_n:
        return 0.0
    return float(pred_n in ctx_n)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def sentence_split(text: str) -> List[str]:
    text = str(text).replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    if not text:
        return []
    parts = re.split(r"(?<=[\.\?!;])\s+", text)
    return [p.strip() for p in parts if p.strip()]

def unique_preserve_order(items: List[str]) -> List[str]:
    out = []
    seen = set()
    for x in items:
        nx = normalize_text(x)
        if nx and nx not in seen:
            out.append(x)
            seen.add(nx)
    return out

def lexical_overlap_count(question: str, text: str) -> int:
    qset = set(question_keywords(question))
    tset = set(tokenize_simple(text))
    return len(qset & tset)

def sentence_score(question: str, sentence: str) -> float:
    qtype = detect_question_type(question)
    qset = set(question_keywords(question))
    toks = tokenize_simple(sentence)
    tset = set(toks)

    score = 0.0
    overlap = len(qset & tset)
    score += 3.0 * overlap

    if qtype in {"when", "numeric"} and re.search(r"\d", sentence):
        score += 2.0
    if qtype == "who" and re.search(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+", sentence):
        score += 1.0
    if qtype == "where" and any(tok in tset for tok in ["city","state","country","located","born","based"]):
        score += 1.0
    if qtype == "yes_no":
        if any(tok in tset for tok in ["american","british","same","different","located","neighborhood","united","states"]):
            score += 1.0

    # shorter sentences are often cleaner evidence
    score -= 0.003 * max(0, len(sentence) - 220)
    return score

def compress_single_context(question: str, context: str, max_sentences: int = 2) -> str:
    sentences = sentence_split(context)
    if not sentences:
        return context.strip()

    scored = [(idx, s, sentence_score(question, s)) for idx, s in enumerate(sentences)]
    scored_sorted = sorted(scored, key=lambda x: x[2], reverse=True)

    kept = [x for x in scored_sorted if x[2] > 0][:max_sentences]
    if not kept:
        kept = scored[:1]

    kept = sorted(kept, key=lambda x: x[0])
    out = " ".join([x[1] for x in kept]).strip()
    return out if out else context.strip()

def prepare_contexts_for_question(
    question: str,
    contexts: List[str],
    top_k: int = 3,
    max_total_chars: int = 3200,
    max_sentences_per_context: int = 2,
) -> List[str]:
    cleaned = []
    for ctx in contexts_to_list(contexts, top_k=999):
        ctx = re.sub(r"\s+", " ", ctx).strip()
        if ctx:
            cleaned.append(ctx)

    cleaned = unique_preserve_order(cleaned)

    # question-aware compression
    if USE_CONTEXT_COMPRESSION:
        compressed = []
        for ctx in cleaned:
            compressed_text = compress_single_context(question, ctx, max_sentences=max_sentences_per_context)
            compressed.append(compressed_text)
        cleaned = compressed

    # re-rank after compression
    scored = []
    for ctx in cleaned:
        score = sentence_score(question, ctx) + 0.1 * lexical_overlap_count(question, ctx)
        scored.append((score, ctx))
    scored.sort(key=lambda x: x[0], reverse=True)

    out = []
    total_chars = 0
    for _, ctx in scored:
        if len(out) >= top_k:
            break
        if total_chars + len(ctx) > max_total_chars and out:
            continue
        out.append(ctx)
        total_chars += len(ctx)

    if not out:
        out = cleaned[:top_k]

    return out[:top_k]

def extract_final_answer(raw_text: str, question: str) -> str:
    text = str(raw_text).strip()

    # Prefer explicit final answer tags
    m = re.search(r"final answer\s*[:\-]\s*(.+)", text, flags=re.IGNORECASE | re.DOTALL)
    if m:
        text = m.group(1).strip()

    # remove any trailing structured sections
    text = re.split(r"\n(?:evidence|reasoning)\s*:", text, flags=re.IGNORECASE)[0].strip()

    # keep the last meaningful line if multi-line
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if lines:
        # prefer a line that starts with answer/final answer, else last line
        tagged = [ln for ln in lines if re.match(r"^(answer|final answer)\s*[:\-]", ln, flags=re.I)]
        text = tagged[-1] if tagged else lines[-1]

    text = re.sub(r"^(answer|final answer)\s*[:\-]\s*", "", text, flags=re.IGNORECASE).strip()
    text = text.strip(" '\"`*")

    if is_abstention(text):
        return "Insufficient evidence."

    qtype = detect_question_type(question)
    norm_yn = normalize_answer_for_ynm(text)
    if qtype == "yes_no" and norm_yn in {"yes", "no", "maybe"}:
        return norm_yn

    # Trim overly long explanatory answers
    if len(text.split()) > 14:
        text = re.split(r"\s+(?:because|since|which|that)\s+", text, maxsplit=1, flags=re.IGNORECASE)[0].strip()
        text = re.split(r"[.;]\s*", text, maxsplit=1)[0].strip()

    return text if text else "Insufficient evidence."

def candidate_answer_score(answer: str, question: str, contexts: List[str]) -> float:
    score = 0.0
    toks = tokenize_simple(answer)
    qtype = detect_question_type(question)

    if not is_abstention(answer):
        score += 4.0
    else:
        score -= 1.0

    if qtype == "yes_no" and normalize_answer_for_ynm(answer) in {"yes", "no", "maybe"}:
        score += 3.0

    if 1 <= len(toks) <= 8:
        score += 1.5
    elif len(toks) > 16:
        score -= 1.0

    score += 1.5 * prediction_in_context(answer, contexts)
    return score


## Oracle dataset adapters (kept for continuity)

These are kept only so the notebook still preserves the **self-contained** workflow from the earlier version.
For the actual final project run, you should normally use **`INPUT_MODE = "retrieval_zip"`**.


In [12]:
from datasets import load_dataset

DATASET_CONFIGS = {
    "hotpotqa": {
        "path": "hotpotqa/hotpot_qa",
        "name": "distractor",
        "split": "validation",
    },
    "natural_questions": {
        "path": "google-research-datasets/natural_questions",
        "split": "validation",
    },
    "pubmedqa": {
        "path": "qiaojin/PubMedQA",
        "name": "pqa_labeled",
        "split": "train",
    },
    "financebench": {
        "path": "PatronusAI/financebench",
        "split": "train",
    },
}

def hotpotqa_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    titles = item.get("context", {}).get("title", [])
    sent_lists = item.get("context", {}).get("sentences", [])
    support_titles = set(item.get("supporting_facts", {}).get("title", []))

    contexts = []
    for title, sentences in zip(titles, sent_lists):
        text = " ".join(sentences).strip()
        contexts.append({
            "title": title,
            "text": text,
            "is_supporting_title": title in support_titles,
        })

    contexts = sorted(contexts, key=lambda x: (not x["is_supporting_title"], x["title"]))
    contexts = contexts_to_list(contexts, top_k=TOP_K_CONTEXTS)

    if not item.get("question") or not item.get("answer"):
        return None

    return {
        "id": item.get("id", ""),
        "domain": "hotpotqa",
        "question": item["question"],
        "answer": item["answer"],
        "contexts": contexts,
        "retrieval_method": "oracle_context",
        "top_k": TOP_K_CONTEXTS,
    }

def pubmedqa_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    contexts_raw = item.get("contexts", item.get("context", []))
    contexts = []
    if isinstance(contexts_raw, list):
        joined = " ".join([str(x) for x in contexts_raw if str(x).strip()])
        if joined:
            contexts = [joined]
    elif isinstance(contexts_raw, str):
        contexts = [contexts_raw]

    question = item.get("question", "")
    answer = item.get("final_decision", "")

    if not question or not answer:
        return None

    return {
        "id": str(item.get("pubid", "")),
        "domain": "pubmedqa",
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list(contexts, top_k=TOP_K_CONTEXTS),
        "retrieval_method": "oracle_context",
        "top_k": TOP_K_CONTEXTS,
    }

def financebench_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    evidence = item.get("evidence", [])
    contexts = []
    for ev in safe_list(evidence):
        if isinstance(ev, dict):
            txt = ev.get("evidence_text", "")
            if txt:
                contexts.append({
                    "title": item.get("doc_name", ""),
                    "text": txt,
                })
        elif isinstance(ev, str) and ev.strip():
            contexts.append(ev)

    question = item.get("question", "")
    answer = item.get("answer", "")

    if not question or not answer:
        return None

    return {
        "id": item.get("financebench_id", ""),
        "domain": "financebench",
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list(contexts, top_k=TOP_K_CONTEXTS),
        "retrieval_method": "oracle_evidence",
        "top_k": TOP_K_CONTEXTS,
    }

def _nq_question_text(item: Dict[str, Any]) -> str:
    q = item.get("question", "")
    if isinstance(q, dict):
        return str(q.get("text", "")).strip()
    return str(q).strip()

def _nq_doc_tokens(item: Dict[str, Any]):
    doc = item.get("document", {})
    toks = doc.get("tokens", {})
    if isinstance(toks, dict):
        token_list = toks.get("token", [])
        is_html = toks.get("is_html", [False] * len(token_list))
        return token_list, is_html
    elif isinstance(toks, list):
        token_list = [str(x.get("token", "")) for x in toks]
        is_html = [bool(x.get("is_html", False)) for x in toks]
        return token_list, is_html
    return [], []

def _nq_slice_tokens(tokens: List[str], is_html: List[bool], start_token: int, end_token: int) -> str:
    start_token = max(0, int(start_token))
    end_token = max(start_token, int(end_token))
    pieces = []
    for tok, html_flag in zip(tokens[start_token:end_token], is_html[start_token:end_token]):
        if not html_flag:
            pieces.append(str(tok))
    text = " ".join(pieces)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def natural_questions_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    question = _nq_question_text(item)
    tokens, is_html = _nq_doc_tokens(item)
    annotations = item.get("annotations", [])

    if not question or not tokens or not annotations:
        return None

    ann = annotations[0]
    answer = ""
    short_answers = ann.get("short_answers", [])
    if short_answers:
        sa = short_answers[0]
        answer = _nq_slice_tokens(tokens, is_html, sa.get("start_token", 0), sa.get("end_token", 0))

    if not answer:
        yes_no = str(ann.get("yes_no_answer", "")).strip().lower()
        if yes_no and yes_no not in {"none", "null"}:
            answer = yes_no

    long_answer = ann.get("long_answer", {})
    context_text = ""
    if isinstance(long_answer, dict):
        context_text = _nq_slice_tokens(
            tokens,
            is_html,
            long_answer.get("start_token", 0),
            long_answer.get("end_token", 0),
        )

    if not answer and context_text:
        answer = context_text[:160]

    if not answer or not context_text:
        return None

    return {
        "id": str(item.get("id", "")),
        "domain": "natural_questions",
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list([context_text], top_k=TOP_K_CONTEXTS),
        "retrieval_method": "oracle_long_answer_context",
        "top_k": TOP_K_CONTEXTS,
    }

ADAPTERS = {
    "hotpotqa": hotpotqa_to_record,
    "natural_questions": natural_questions_to_record,
    "pubmedqa": pubmedqa_to_record,
    "financebench": financebench_to_record,
}

def load_oracle_dataset(domain: str, sample_size: Optional[int] = None) -> pd.DataFrame:
    cfg = DATASET_CONFIGS[domain]
    adapter = ADAPTERS[domain]
    sample_size = sample_size or SAMPLES_PER_DOMAIN.get(domain, 10)

    kwargs = {"path": cfg["path"], "split": cfg["split"]}
    if "name" in cfg:
        kwargs["name"] = cfg["name"]

    records = []

    if domain == "natural_questions" and USE_STREAMING_FOR_NQ:
        ds = load_dataset(**kwargs, streaming=True)
        for item in ds:
            rec = adapter(item)
            if rec is not None:
                records.append(rec)
            if len(records) >= sample_size:
                break
    else:
        ds = load_dataset(**kwargs)
        sample_size = min(sample_size, len(ds))
        ds = ds.select(range(sample_size))
        for item in ds:
            rec = adapter(item)
            if rec is not None:
                records.append(rec)

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f"No valid records loaded for domain={domain}")

    df["question"] = df["question"].fillna("").astype(str)
    df["answer"] = df["answer"].fillna("").astype(str)
    df["contexts"] = df["contexts"].apply(lambda x: contexts_to_list(x, top_k=TOP_K_CONTEXTS))
    return df


## Team retrieval loaders

This notebook can now read your teammate's **retrieval zip outputs directly**.

### Notes
- The uploaded retrieval zip is the main usable input for generation.
- The uploaded preprocess zip is treated as **optional reference** only.
- If the preprocess zip contains Git-LFS pointer files instead of real artifacts, generation still works because it only needs the retrieval outputs.


In [13]:
def load_retrieval_jsonl(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"JSONL file not found: {path}")

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "id": obj.get("id", ""),
                "domain": obj.get("domain", "unknown"),
                "question": obj.get("question", ""),
                "answer": obj.get("answer", ""),
                "contexts": contexts_to_list(obj.get("contexts", []), top_k=obj.get("top_k", TOP_K_CONTEXTS)),
                "retrieval_method": obj.get("retrieval_method", "unknown"),
                "top_k": obj.get("top_k", TOP_K_CONTEXTS),
                "source_file": str(path.name),
            })

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("The JSONL file was read successfully, but no valid records were found.")
    return df

def _extract_text_from_result_item(item: Dict[str, Any]) -> str:
    return str(item.get("chunk_text", item.get("text", ""))).strip()

def _extract_title_from_result_item(item: Dict[str, Any]) -> str:
    meta = item.get("metadata", {}) or {}
    return str(meta.get("title", meta.get("chunk_id", item.get("doc_id", "")))).strip()

def _extract_answer_from_result_item(item: Dict[str, Any]) -> str:
    meta = item.get("metadata", {}) or {}
    return str(meta.get("answer", "")).strip()

def _extract_domain_from_result_item(item: Dict[str, Any]) -> str:
    meta = item.get("metadata", {}) or {}
    return str(meta.get("dataset", meta.get("source", "unknown"))).strip()

def _extract_record_id(obj: Dict[str, Any], file_name: str) -> str:
    if isinstance(obj, dict) and obj.get("results"):
        meta = obj["results"][0].get("metadata", {}) or {}
        if "original_index" in meta:
            return f"{_extract_domain_from_result_item(obj['results'][0])}_{meta['original_index']}"
    return Path(file_name).stem

def _query_result_to_record(obj: Dict[str, Any], file_name: str, retrieval_method_label: str) -> Optional[Dict[str, Any]]:
    if not isinstance(obj, dict):
        return None
    if "query" not in obj or "results" not in obj:
        return None
    results = safe_list(obj.get("results", []))
    if not results:
        return None

    question = str(obj.get("query", "")).strip()
    answer = _extract_answer_from_result_item(results[0])
    domain = _extract_domain_from_result_item(results[0]) or "unknown"

    contexts = []
    for r in results:
        text = _extract_text_from_result_item(r)
        if not text:
            continue
        title = _extract_title_from_result_item(r)
        contexts.append({
            "title": title,
            "text": text,
            "score": r.get("score", None),
        })

    if not question or not contexts:
        return None

    return {
        "id": _extract_record_id(obj, file_name),
        "domain": domain,
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list(contexts, top_k=999),
        "retrieval_method": retrieval_method_label,
        "top_k": int(obj.get("top_k", len(contexts))),
        "source_file": file_name,
    }

def _is_probably_relevant_dense_chunk(question: str, text: str) -> bool:
    overlap = lexical_overlap_count(question, text)
    qkw = question_keywords(question)
    if overlap >= 2:
        return True
    if overlap >= 1 and len(qkw) <= 3:
        return True
    return False

def load_retrieval_zip(path: str | Path, variant: str = "bm25") -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Retrieval zip not found: {path}")

    with zipfile.ZipFile(path, "r") as zf:
        names = [n for n in zf.namelist() if n.endswith(".json")]

        bm25_files = sorted([n for n in names if "/result/" in n and "retrieval_results_bm25_" in n])
        dense_files = sorted([n for n in names if "/dense/" in n and "retrieval_results_dense_" in n])

        if variant == "bm25":
            selected_files = bm25_files
            records = []
            for fn in selected_files:
                obj = json.loads(zf.read(fn))
                rec = _query_result_to_record(obj, fn, "bm25")
                if rec is not None:
                    records.append(rec)

        elif variant == "dense":
            selected_files = dense_files
            records = []
            for fn in selected_files:
                obj = json.loads(zf.read(fn))
                rec = _query_result_to_record(obj, fn, "dense")
                if rec is not None:
                    records.append(rec)

        elif variant == "hybrid_safe":
            bm25_records = {}
            dense_records = {}

            for fn in bm25_files:
                obj = json.loads(zf.read(fn))
                rec = _query_result_to_record(obj, fn, "bm25")
                if rec is not None:
                    bm25_records[rec["question"]] = rec

            for fn in dense_files:
                obj = json.loads(zf.read(fn))
                rec = _query_result_to_record(obj, fn, "dense")
                if rec is not None:
                    dense_records[rec["question"]] = rec

            all_questions = list(dict.fromkeys(list(bm25_records.keys()) + list(dense_records.keys())))
            records = []

            for q in all_questions:
                if q in bm25_records:
                    base = dict(bm25_records[q])
                else:
                    base = dict(dense_records[q])

                combined_contexts = list(base["contexts"])

                # Append at most one dense chunk if it looks relevant and non-duplicate
                if q in dense_records and q in bm25_records:
                    dense_ctxs = dense_records[q]["contexts"]
                    for ctx in dense_ctxs:
                        if normalize_text(ctx) in {normalize_text(x) for x in combined_contexts}:
                            continue
                        if _is_probably_relevant_dense_chunk(q, ctx):
                            combined_contexts.append(ctx)
                            break

                base["contexts"] = combined_contexts[:MAX_CONTEXTS_AFTER_HYBRID]
                base["retrieval_method"] = "hybrid_safe"
                records.append(base)
        else:
            raise ValueError("variant must be 'bm25', 'dense', or 'hybrid_safe'")

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("No valid retrieval records were parsed from the zip.")
    return df

def inspect_preprocess_zip(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        return pd.DataFrame([{"path": str(path), "status": "missing"}])

    rows = []
    with zipfile.ZipFile(path, "r") as zf:
        for name in sorted(zf.namelist()):
            if name.endswith("/"):
                continue
            size = zf.getinfo(name).file_size
            head = zf.read(name)[:120].decode("utf-8", errors="ignore")
            rows.append({
                "file": name,
                "size_bytes": size,
                "looks_like_git_lfs_pointer": head.startswith("version https://git-lfs.github.com/spec/v1"),
            })
    return pd.DataFrame(rows)


## Build the evaluation dataframe

In [14]:
if INPUT_MODE == "oracle":
    frames = []
    for domain in DOMAINS_TO_RUN:
        print(f"Loading oracle data for: {domain}")
        frames.append(load_oracle_dataset(domain, sample_size=SAMPLES_PER_DOMAIN.get(domain, 10)))
    eval_df = pd.concat(frames, ignore_index=True)

elif INPUT_MODE == "team_jsonl":
    eval_df = load_retrieval_jsonl(TEAM_JSONL_PATH)

elif INPUT_MODE == "retrieval_zip":
    eval_df = load_retrieval_zip(RETRIEVAL_ZIP_PATH, variant=RETRIEVAL_VARIANT)

else:
    raise ValueError("INPUT_MODE must be 'retrieval_zip', 'team_jsonl', or 'oracle'")

print("Evaluation dataframe shape:", eval_df.shape)
display(eval_df.head(3))


Evaluation dataframe shape: (20, 8)


,id,domain,question,answer,contexts,retrieval_method,top_k,source_file
0,hotpotqa_0,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[[hotpotqa_0_1] Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film c...",bm25,3,RAGProjectRetrive-main/result/retrieval_results_bm25_3_1.json
1,hotpotqa_1608,hotpotqa,Are Local H and For Against both from the United States?,The Bears,"[[hotpotqa_1608_21928] h its main campus in Macon, Georgia, United States., [hotpotqa_5235_71561] The Team EP: The Team EP (stylized as The TEAM ep) is an extended play (EP) by American alternativ...",bm25,3,RAGProjectRetrive-main/result/retrieval_results_bm25_3_10.json
2,hotpotqa_10,hotpotqa,"What is the name of the fight song of the university whose main campus is in Lawrence, Kansas and whose branch campuses are in the Kansas City metropolitan area?",Kansas Song,"[[hotpotqa_10_166] referred to as KU or Kansas, is a public research university in the U.S. state of Kansas. The main campus in Lawrence, one of the largest college towns in Kansas, is on Mount Or...",bm25,3,RAGProjectRetrive-main/result/retrieval_results_bm25_3_11.json


## Optional: inspect the uploaded preprocess zip

This is mainly to verify whether the uploaded preprocess archive contains the real large artifacts or only Git-LFS pointers.
Generation does **not** depend on this to run.


In [ ]:
if PREPROCESS_ZIP_PATH.exists():
    preprocess_inspect_df = inspect_preprocess_zip(PREPROCESS_ZIP_PATH)
    display(preprocess_inspect_df.head(10))
else:
    print("Preprocess zip not found at:", PREPROCESS_ZIP_PATH)


## Preview question-aware context preparation

This is one of the main quality improvements in this version.
It compresses and re-ranks retrieved chunks before they are fed to the base model.


In [15]:
preview_df = eval_df.head(5).copy()
preview_df["prepared_contexts"] = preview_df.apply(
    lambda row: prepare_contexts_for_question(
        question=row["question"],
        contexts=row["contexts"],
        top_k=TOP_K_CONTEXTS,
        max_total_chars=MAX_TOTAL_CONTEXT_CHARS,
        max_sentences_per_context=MAX_SENTENCES_PER_CONTEXT,
    ),
    axis=1,
)
display(preview_df[["question", "answer", "retrieval_method", "prepared_contexts"]])


,question,answer,retrieval_method,prepared_contexts
0,Were Scott Derrickson and Ed Wood of the same nationality?,yes,bm25,"[Ed Wood: Edward Davis Wood Jr. Deliver Us from Evil (2014 film): Deliver Us from Evil is a 2014 American supernatural horror film directed by Scott Derrickson and produced by Jerry Bruckheimer., ..."
1,Are Local H and For Against both from the United States?,The Bears,bm25,"[[hotpotqa_1608_21928] h its main campus in Macon, Georgia, United States., [hotpotqa_4901_67001] h descent to run for Congress in the United States., [hotpotqa_5235_71561] The Team EP: The Team E..."
2,"What is the name of the fight song of the university whose main campus is in Lawrence, Kansas and whose branch campuses are in the Kansas City metropolitan area?",Kansas Song,bm25,"[The main campus in Lawrence, one of the largest college towns in Kansas, is on Mount Oread, the highest elevation in Lawrence. Two branch campuses are in the Kansas City metropolitan area: the Ed..."
3,"What screenwriter with credits for ""Evolution"" co-wrote a film starring Nicolas Cage and Téa Leoni?",David Weissman,bm25,"[[hotpotqa_11_174] His film credits include ""The Family Man"" (2000), ""Evolution"" (2001), and """"When in Rome"""" (2010). The film stars Michael Biehn, Nicolas Cage, Charlie Sheen, James Coburn, and P..."
4,What year did Guns N Roses perform a promo for a movie starring Arnold Schwarzenegger as a former New York Police detective?,1999,bm25,"[End of Days (film): End of Days is a 1999 American fantasy action horror thriller film directed by Peter Hyams and starring Arnold Schwarzenegger, Gabriel Byrne, Robin Tunney, Kevin Pollak, Rod S..."


## Prompt design

This merged notebook keeps the **stricter base-model prompt + fallback prompt** from the final base notebook,
and also restores the **instruction / chat prompt** path from the teammate notebook.

For the comparison section:

- the **base model** uses the improved zero-shot prompt with fallback
- the **instruction-tuned model** uses a structured chat-style prompt
- both use the same prepared retrieval contexts


In [ ]:
def build_base_prompt(question: str, contexts: List[str]) -> str:
    context_block = join_contexts(contexts)
    qtype = detect_question_type(question)

    if qtype == "yes_no":
        answer_rule = "If the evidence supports yes/no, output exactly yes or no."
    else:
        answer_rule = "Output the shortest possible answer phrase, not a full explanatory sentence."

    return f"""You are a careful question answering system.
Use only the retrieved evidence below.

Question type: {qtype}
Question: {question}

Retrieved evidence:
{context_block}

Rules:
1. Use only the evidence above.
2. If the evidence is not sufficient, output exactly: Insufficient evidence.
3. {answer_rule}
4. Do not add background knowledge.
5. Keep the final answer extremely concise.

Output format:
Evidence: <brief supporting phrase(s) or none>
Final answer: <answer>"""

def build_base_fallback_prompt(question: str, contexts: List[str]) -> str:
    context_block = join_contexts(contexts)
    qtype = detect_question_type(question)

    if qtype == "yes_no":
        final_line = "Final answer: yes or no or Insufficient evidence."
    else:
        final_line = "Final answer: the shortest answer phrase possible, or Insufficient evidence."

    return f"""Answer strictly from the evidence.

Question: {question}

Evidence:
{context_block}

Return only two lines:
Evidence: <minimal supporting span(s) or none>
{final_line}"""


def build_instruct_messages(question: str, contexts: List[str]) -> List[Dict[str, str]]:
    context_block = join_contexts(contexts)
    qtype = detect_question_type(question)

    if qtype == "yes_no":
        answer_rule = "If the evidence supports yes/no, answer with exactly yes or no."
    else:
        answer_rule = "Return the shortest direct answer phrase, not a long explanation."

    return [
        {
            "role": "system",
            "content": (
                "You are a careful question answering assistant. "
                "Use only the retrieved evidence. "
                "Do not use outside knowledge. "
                "If the evidence is insufficient, reply exactly: Insufficient evidence. "
                + answer_rule
            ),
        },
        {
            "role": "user",
            "content": (
                f"Question: {question}\n\n"
                f"Retrieved evidence:\n{context_block}\n\n"
                "Return a very concise final answer."
            ),
        },
    ]


## Model loading

In [17]:
def pick_compute_dtype():
    if not torch.cuda.is_available() or FORCE_CPU:
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

COMPUTE_DTYPE = pick_compute_dtype()
print("COMPUTE_DTYPE =", COMPUTE_DTYPE)

def make_quantization_config():
    if not USE_4BIT or not torch.cuda.is_available() or FORCE_CPU:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

def load_model_and_tokenizer(model_name: str):
    clear_memory()
    quant_config = make_quantization_config()

    model_kwargs = {
        "low_cpu_mem_usage": True,
    }

    if quant_config is not None:
        model_kwargs["quantization_config"] = quant_config
        model_kwargs["device_map"] = "auto"
    else:
        if torch.cuda.is_available() and not FORCE_CPU:
            model_kwargs["torch_dtype"] = COMPUTE_DTYPE
            model_kwargs["device_map"] = "auto"
        else:
            model_kwargs["torch_dtype"] = torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    return tokenizer, model

def get_inference_device():
    if torch.cuda.is_available() and not FORCE_CPU:
        return "cuda"
    return "cpu"


COMPUTE_DTYPE = torch.float16


## Generation core

The shared generation helpers below support both:

- the **improved base-final path**
- the preserved **base vs instruct comparison** path

The base path keeps its fallback prompt and answer-selection heuristics.
The instruct path uses chat prompting but shares the same prepared retrieval contexts.


In [ ]:

@torch.inference_mode()
def _generate_text(model, tokenizer, prompt: str) -> str:
    device = get_inference_device()
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = generated[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

@torch.inference_mode()
def _generate_chat_text(model, tokenizer, messages: List[Dict[str, str]]) -> str:
    device = get_inference_device()
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = generated[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

@torch.inference_mode()
def generate_one_answer(model, tokenizer, question: str, contexts: List[str], prompt_mode: str = "base") -> Dict[str, Any]:
    prepared_contexts = prepare_contexts_for_question(
        question=question,
        contexts=contexts,
        top_k=TOP_K_CONTEXTS,
        max_total_chars=MAX_TOTAL_CONTEXT_CHARS,
        max_sentences_per_context=MAX_SENTENCES_PER_CONTEXT,
    )

    if prompt_mode == "base":
        prompt_1 = build_base_prompt(question, prepared_contexts)
        raw_1 = _generate_text(model, tokenizer, prompt_1)
        ans_1 = extract_final_answer(raw_1, question)

        candidates = [{
            "prompt_name": "primary",
            "raw_output": raw_1,
            "parsed_answer": ans_1,
            "score": candidate_answer_score(ans_1, question, prepared_contexts),
        }]

        if USE_FALLBACK_PROMPT:
            needs_retry = (
                is_abstention(ans_1)
                or len(tokenize_simple(ans_1)) > 10
                or (detect_question_type(question) == "yes_no" and normalize_answer_for_ynm(ans_1) not in {"yes", "no", "maybe"})
            )
            if needs_retry:
                prompt_2 = build_base_fallback_prompt(question, prepared_contexts)
                raw_2 = _generate_text(model, tokenizer, prompt_2)
                ans_2 = extract_final_answer(raw_2, question)
                candidates.append({
                    "prompt_name": "fallback",
                    "raw_output": raw_2,
                    "parsed_answer": ans_2,
                    "score": candidate_answer_score(ans_2, question, prepared_contexts),
                })

        best = sorted(candidates, key=lambda x: x["score"], reverse=True)[0]
        return {
            "prediction": best["parsed_answer"],
            "best_prompt_name": best["prompt_name"],
            "raw_model_output": best["raw_output"],
            "prepared_contexts": prepared_contexts,
            "candidate_answers": candidates,
        }

    elif prompt_mode == "instruct":
        messages = build_instruct_messages(question, prepared_contexts)
        raw = _generate_chat_text(model, tokenizer, messages)
        ans = extract_final_answer(raw, question)
        candidate = {
            "prompt_name": "chat_primary",
            "raw_output": raw,
            "parsed_answer": ans,
            "score": candidate_answer_score(ans, question, prepared_contexts),
        }
        return {
            "prediction": ans,
            "best_prompt_name": candidate["prompt_name"],
            "raw_model_output": raw,
            "prepared_contexts": prepared_contexts,
            "candidate_answers": [candidate],
        }

    else:
        raise ValueError("prompt_mode must be 'base' or 'instruct'")

def run_model_on_eval_df(
    eval_df: pd.DataFrame,
    model_name: str,
    model_label: str,
    prompt_mode: str = "base",
) -> pd.DataFrame:
    tokenizer, model = load_model_and_tokenizer(model_name)

    rows = []
    iterator = tqdm(eval_df.to_dict("records"), total=len(eval_df), desc=f"Running {model_label}")

    for row in iterator:
        out = generate_one_answer(
            model=model,
            tokenizer=tokenizer,
            question=row["question"],
            contexts=row["contexts"],
            prompt_mode=prompt_mode,
        )

        rows.append({
            **row,
            "model_label": model_label,
            "model_name": model_name,
            "prompt_mode": prompt_mode,
            "prediction": out["prediction"],
            "best_prompt_name": out["best_prompt_name"],
            "raw_model_output": out["raw_model_output"],
            "prepared_contexts": out["prepared_contexts"],
            "candidate_answers": json.dumps(out["candidate_answers"], ensure_ascii=False),
        })

    del model
    del tokenizer
    clear_memory()

    return pd.DataFrame(rows)


## Run the base model generation

In [18]:
@torch.inference_mode()
def _generate_text(model, tokenizer, prompt: str) -> str:
    device = get_inference_device()
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = generated[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

@torch.inference_mode()
def generate_one_answer(model, tokenizer, question: str, contexts: List[str]) -> Dict[str, Any]:
    prepared_contexts = prepare_contexts_for_question(
        question=question,
        contexts=contexts,
        top_k=TOP_K_CONTEXTS,
        max_total_chars=MAX_TOTAL_CONTEXT_CHARS,
        max_sentences_per_context=MAX_SENTENCES_PER_CONTEXT,
    )

    prompt_1 = build_base_prompt(question, prepared_contexts)
    raw_1 = _generate_text(model, tokenizer, prompt_1)
    ans_1 = extract_final_answer(raw_1, question)

    candidates = [{
        "prompt_name": "primary",
        "raw_output": raw_1,
        "parsed_answer": ans_1,
        "score": candidate_answer_score(ans_1, question, prepared_contexts),
    }]

    if USE_FALLBACK_PROMPT:
        needs_retry = (
            is_abstention(ans_1)
            or len(tokenize_simple(ans_1)) > 10
            or (detect_question_type(question) == "yes_no" and normalize_answer_for_ynm(ans_1) not in {"yes", "no", "maybe"})
        )
        if needs_retry:
            prompt_2 = build_base_fallback_prompt(question, prepared_contexts)
            raw_2 = _generate_text(model, tokenizer, prompt_2)
            ans_2 = extract_final_answer(raw_2, question)
            candidates.append({
                "prompt_name": "fallback",
                "raw_output": raw_2,
                "parsed_answer": ans_2,
                "score": candidate_answer_score(ans_2, question, prepared_contexts),
            })

    best = sorted(candidates, key=lambda x: x["score"], reverse=True)[0]

    return {
        "prediction": best["parsed_answer"],
        "best_prompt_name": best["prompt_name"],
        "raw_model_output": best["raw_output"],
        "prepared_contexts": prepared_contexts,
        "candidate_answers": candidates,
    }

def run_model_on_eval_df(eval_df: pd.DataFrame, model_name: str, model_label: str) -> pd.DataFrame:
    tokenizer, model = load_model_and_tokenizer(model_name)

    rows = []
    iterator = tqdm(eval_df.to_dict("records"), total=len(eval_df), desc=f"Running {model_label}")

    for row in iterator:
        out = generate_one_answer(
            model=model,
            tokenizer=tokenizer,
            question=row["question"],
            contexts=row["contexts"],
        )

        rows.append({
            **row,
            "model_label": model_label,
            "model_name": model_name,
            "prediction": out["prediction"],
            "best_prompt_name": out["best_prompt_name"],
            "raw_model_output": out["raw_model_output"],
            "prepared_contexts": out["prepared_contexts"],
            "candidate_answers": json.dumps(out["candidate_answers"], ensure_ascii=False),
        })

    del model
    del tokenizer
    clear_memory()

    return pd.DataFrame(rows)


In [19]:
base_results = run_model_on_eval_df(
    eval_df=eval_df,
    model_name=BASE_MODEL_NAME,
    model_label="base_zero_shot_final",
)

display(base_results.head(3))


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Running base_zero_shot_final:   0%|          | 0/20 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,id,domain,question,answer,contexts,retrieval_method,top_k,source_file,model_label,model_name,prediction,best_prompt_name,raw_model_output,prepared_contexts,candidate_answers
0,hotpotqa_0,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[[hotpotqa_0_1] Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film c...",bm25,3,RAGProjectRetrive-main/result/retrieval_results_bm25_3_1.json,base_zero_shot_final,Qwen/Qwen2.5-7B,yes,primary,Yes,"[Ed Wood: Edward Davis Wood Jr. Deliver Us from Evil (2014 film): Deliver Us from Evil is a 2014 American supernatural horror film directed by Scott Derrickson and produced by Jerry Bruckheimer., ...","[{""prompt_name"": ""primary"", ""raw_output"": ""Yes"", ""parsed_answer"": ""yes"", ""score"": 8.5}]"
1,hotpotqa_1608,hotpotqa,Are Local H and For Against both from the United States?,The Bears,"[[hotpotqa_1608_21928] h its main campus in Macon, Georgia, United States., [hotpotqa_5235_71561] The Team EP: The Team EP (stylized as The TEAM ep) is an extended play (EP) by American alternativ...",bm25,3,RAGProjectRetrive-main/result/retrieval_results_bm25_3_10.json,base_zero_shot_final,Qwen/Qwen2.5-7B,yes,primary,Yes\n\nEvidence: The Team Ep: The Team ep is an extended-play (ep) by american alternative rock band local h.\nFinal answer: Yes,"[[hotpotqa_1608_21928] h its main campus in Macon, Georgia, United States., [hotpotqa_4901_67001] h descent to run for Congress in the United States., [hotpotqa_5235_71561] The Team EP: The Team E...","[{""prompt_name"": ""primary"", ""raw_output"": ""Yes\n\nEvidence: The Team Ep: The Team ep is an extended-play (ep) by american alternative rock band local h.\nFinal answer: Yes"", ""parsed_answer"": ""yes""..."
2,hotpotqa_10,hotpotqa,"What is the name of the fight song of the university whose main campus is in Lawrence, Kansas and whose branch campuses are in the Kansas City metropolitan area?",Kansas Song,"[[hotpotqa_10_166] referred to as KU or Kansas, is a public research university in the U.S. state of Kansas. The main campus in Lawrence, one of the largest college towns in Kansas, is on Mount Or...",bm25,3,RAGProjectRetrive-main/result/retrieval_results_bm25_3_11.json,base_zero_shot_final,Qwen/Qwen2.5-7B,Jayhawk,primary,Evidence: main campus in Kansas\n\nFinal answer: Jayhawk,"[The main campus in Lawrence, one of the largest college towns in Kansas, is on Mount Oread, the highest elevation in Lawrence. Two branch campuses are in the Kansas City metropolitan area: the Ed...","[{""prompt_name"": ""primary"", ""raw_output"": ""Evidence: main campus in Kansas\n\nFinal answer: Jayhawk"", ""parsed_answer"": ""Jayhawk"", ""score"": 5.5}]"


## Score results

If `answer` is available in the retrieval metadata, this section computes metrics.
If not, the notebook still saves raw predictions for qualitative analysis.


In [20]:
all_results = base_results.copy()

has_gold = "answer" in all_results.columns and all_results["answer"].fillna("").astype(str).str.len().gt(0).any()

if has_gold:
    all_results["exact_match"] = all_results.apply(
        lambda row: exact_match_score(row["prediction"], row["answer"]),
        axis=1,
    )
    all_results["token_f1"] = all_results.apply(
        lambda row: token_f1_score(row["prediction"], row["answer"]),
        axis=1,
    )
    all_results["abstained"] = all_results["prediction"].apply(is_abstention).astype(float)
    all_results["prediction_in_context"] = all_results.apply(
        lambda row: prediction_in_context(row["prediction"], row["prepared_contexts"]),
        axis=1,
    )

    summary_by_domain = (
        all_results
        .groupby(["domain", "model_label", "retrieval_method"], as_index=False)
        .agg(
            n=("id", "count"),
            em=("exact_match", "mean"),
            f1=("token_f1", "mean"),
            abstain_rate=("abstained", "mean"),
            pred_in_context_rate=("prediction_in_context", "mean"),
        )
        .sort_values(["domain", "model_label", "retrieval_method"])
    )

    summary_overall = (
        all_results
        .groupby(["model_label", "retrieval_method"], as_index=False)
        .agg(
            n=("id", "count"),
            em=("exact_match", "mean"),
            f1=("token_f1", "mean"),
            abstain_rate=("abstained", "mean"),
            pred_in_context_rate=("prediction_in_context", "mean"),
        )
        .sort_values(["model_label", "retrieval_method"])
    )

    display(summary_by_domain)
    display(summary_overall)
else:
    summary_by_domain = pd.DataFrame()
    summary_overall = pd.DataFrame()
    print("No gold answers available in the loaded inputs, so only raw outputs will be saved.")


,domain,model_label,retrieval_method,n,em,f1,abstain_rate,pred_in_context_rate
0,hotpotqa,base_zero_shot_final,bm25,20,0.1,0.219167,0.1,0.25


,model_label,retrieval_method,n,em,f1,abstain_rate,pred_in_context_rate
0,base_zero_shot_final,bm25,20,0.1,0.219167,0.1,0.25


## Save outputs

In [21]:
raw_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_raw_results.csv"
jsonl_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_raw_results.jsonl"
domain_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_summary_by_domain.csv"
overall_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_summary_overall.csv"

all_results.to_csv(raw_csv_path, index=False)

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in all_results.to_dict("records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

if not summary_by_domain.empty:
    summary_by_domain.to_csv(domain_csv_path, index=False)
if not summary_overall.empty:
    summary_overall.to_csv(overall_csv_path, index=False)

print("Saved:")
print("-", raw_csv_path)
print("-", jsonl_path)
if not summary_by_domain.empty:
    print("-", domain_csv_path)
if not summary_overall.empty:
    print("-", overall_csv_path)


Saved:
- /content/cs6493_topic4_generation_base_final/outputs/qwen_7b_base_generation_final_raw_results.csv
- /content/cs6493_topic4_generation_base_final/outputs/qwen_7b_base_generation_final_raw_results.jsonl
- /content/cs6493_topic4_generation_base_final/outputs/qwen_7b_base_generation_final_summary_by_domain.csv
- /content/cs6493_topic4_generation_base_final/outputs/qwen_7b_base_generation_final_summary_overall.csv


## Quick qualitative inspection

In [22]:
display(
    all_results[
        [
            "id",
            "domain",
            "question",
            "answer",
            "retrieval_method",
            "prediction",
            "best_prompt_name",
            "prepared_contexts",
            "raw_model_output",
        ]
    ].head(10)
)


,id,domain,question,answer,retrieval_method,prediction,best_prompt_name,prepared_contexts,raw_model_output
0,hotpotqa_0,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,bm25,yes,primary,"[Ed Wood: Edward Davis Wood Jr. Deliver Us from Evil (2014 film): Deliver Us from Evil is a 2014 American supernatural horror film directed by Scott Derrickson and produced by Jerry Bruckheimer., ...",Yes
1,hotpotqa_1608,hotpotqa,Are Local H and For Against both from the United States?,The Bears,bm25,yes,primary,"[[hotpotqa_1608_21928] h its main campus in Macon, Georgia, United States., [hotpotqa_4901_67001] h descent to run for Congress in the United States., [hotpotqa_5235_71561] The Team EP: The Team E...",Yes\n\nEvidence: The Team Ep: The Team ep is an extended-play (ep) by american alternative rock band local h.\nFinal answer: Yes
2,hotpotqa_10,hotpotqa,"What is the name of the fight song of the university whose main campus is in Lawrence, Kansas and whose branch campuses are in the Kansas City metropolitan area?",Kansas Song,bm25,Jayhawk,primary,"[The main campus in Lawrence, one of the largest college towns in Kansas, is on Mount Oread, the highest elevation in Lawrence. Two branch campuses are in the Kansas City metropolitan area: the Ed...",Evidence: main campus in Kansas\n\nFinal answer: Jayhawk
3,hotpotqa_11,hotpotqa,"What screenwriter with credits for ""Evolution"" co-wrote a film starring Nicolas Cage and Téa Leoni?",David Weissman,bm25,David Diamond,primary,"[[hotpotqa_11_174] His film credits include ""The Family Man"" (2000), ""Evolution"" (2001), and """"When in Rome"""" (2010). The film stars Michael Biehn, Nicolas Cage, Charlie Sheen, James Coburn, and P...","Evidence: ""Evolution"", ""Time to Kill""\nFinal answer: David Diamond"
4,hotpotqa_12,hotpotqa,What year did Guns N Roses perform a promo for a movie starring Arnold Schwarzenegger as a former New York Police detective?,1999,bm25,1979,primary,"[End of Days (film): End of Days is a 1999 American fantasy action horror thriller film directed by Peter Hyams and starring Arnold Schwarzenegger, Gabriel Byrne, Robin Tunney, Kevin Pollak, Rod S...","Evidence: ""End of days"" (Guns n' Roses song)\nFinal answer: 1979"
5,hotpotqa_13,hotpotqa,Are Random House Tower and 888 7th Avenue both used for real estate?,no,bm25,yes,primary,"[[hotpotqa_13_201] d States, that is used as the headquarters of book publisher Random House and a luxury apartment complex. Pershing Square Capital Management: Pershing Square Capital Management ...",Yes\n\nEvidence: PershING Square Capital Management...is located at 7th avenue...\nFinal answer: yes
6,hotpotqa_14,hotpotqa,The football manager who recruited David Beckham managed Manchester United during what timeframe?,from 1986 to 2013,bm25,1st half of 19th century,primary,"[[hotpotqa_14_230] Academy: The David Beckham Academy was a football school founded by England international David Beckham in 2005. 1995–96 Manchester United F.C., [hotpotqa_14_229] ur of Sir Matt...",Evidence: 195-196 Manchester united\nFinal answer: 1st half of 19th century
7,hotpotqa_15,hotpotqa,Brown State Fishing Lake is in a country that has a population of how many inhabitants ?,"9,984",bm25,- Document,primary,"[Brown State Fishing Lake: Brown State Fishing Lake (sometimes also known as Brown State Fishing Lake And Wildlife Area) is a protected area in Brown County, Kansas in the United States. Neosho St...","To answer the question about the population of Brown State Fishing lake's country, I will follow these steps:\n\nStep 1: Identify relevant information from the given documents.\n- Document 1 menti..."
8,hotpotqa_16,hotpotqa,The Vermont Catamounts men's soccer team currently competes in a conference that was formerly known as what from 1988 to 1996?,the North Atlantic Conference,bm25,Insufficient evidence.,primary,[The conference was known as the Eastern College Athletic Conference-North from 1979 to 1988 and the North Atlantic Conference from 1988 t

## Notes for your final project submission

This version is specifically tuned for the **base model generation** role:

- it uses the **real retrieval outputs** instead of relying on large dataset downloads
- it is **BM25-first** by default because the uploaded dense retrieval examples can be noisy
- it adds **question-aware context compression**
- it uses a **stricter parseable prompt**
- it includes a **fallback prompt** to reduce over-long or overly conservative base-model outputs

If you later want to try a slightly more aggressive setting, you can change:

- `RETRIEVAL_VARIANT = "hybrid_safe"`
- `TOP_K_CONTEXTS = 4`
- `MODEL_PRESET = "qwen_3b"` if Colab memory is tight


## Preserved teammate comparison section

The cells below preserve the teammate notebook's **base vs instruct comparison block** and its saved outputs.
To avoid conflicts with the base-final workflow above, the result variables are renamed with a `comparison_` prefix.


## Run the comparison

The next two cells do the actual experiment:

- zero-shot/base model
- instruction-tuned model


In [16]:

comparison_base_results = run_model_on_eval_df(
    eval_df=eval_df,
    model_name=BASE_MODEL_NAME,
    model_label="base_zero_shot",
    prompt_mode="base",
)

display(comparison_base_results.head(3))


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Running base_zero_shot:   0%|          | 0/8 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,id,domain,question,answer,contexts,retrieval_method,top_k,source_file,model_label,model_name,prompt_mode,prediction
0,0,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film concerns the per...",bm25,3,retrieval_results_bm25_3_1.json,base_zero_shot,Qwen/Qwen2.5-7B,base,Insufficient evidence.
1,11,hotpotqa,"What screenwriter with credits for ""Evolution"" co-wrote a film starring Nicolas Cage and Téa Leoni?",David Weissman,"[His film credits include ""The Family Man"" (2000), ""Evolution"" (2001), and """"When in Rome"""" (2010). Deadfall (1993 film): Deadfall is a 1993 crime drama film directed by Christopher Coppola. Coppo...",bm25,3,retrieval_results_bm25_3_12.json,base_zero_shot,Qwen/Qwen2.5-7B,base,Insufficient evidence.
2,12,hotpotqa,What year did Guns N Roses perform a promo for a movie starring Arnold Schwarzenegger as a former New York Police detective?,1999,"[ollows former New York Police Department detective Jericho Cane (Schwarzenegger) after he saves a banker (Byrne) from an assassin, finds himself embroiled in a religious conflict, and must protec...",bm25,3,retrieval_results_bm25_3_13.json,base_zero_shot,Qwen/Qwen2.5-7B,base,1999


In [17]:

comparison_instruct_results = run_model_on_eval_df(
    eval_df=eval_df,
    model_name=INSTRUCT_MODEL_NAME,
    model_label="instruction_tuned",
    prompt_mode="instruct",
)

display(comparison_instruct_results.head(3))


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Running instruction_tuned:   0%|          | 0/8 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,id,domain,question,answer,contexts,retrieval_method,top_k,source_file,model_label,model_name,prompt_mode,prediction
0,0,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film concerns the per...",bm25,3,retrieval_results_bm25_3_1.json,instruction_tuned,Qwen/Qwen2.5-7B-Instruct,instruct,Yes.
1,11,hotpotqa,"What screenwriter with credits for ""Evolution"" co-wrote a film starring Nicolas Cage and Téa Leoni?",David Weissman,"[His film credits include ""The Family Man"" (2000), ""Evolution"" (2001), and """"When in Rome"""" (2010). Deadfall (1993 film): Deadfall is a 1993 crime drama film directed by Christopher Coppola. Coppo...",bm25,3,retrieval_results_bm25_3_12.json,instruction_tuned,Qwen/Qwen2.5-7B-Instruct,instruct,David Diamond
2,12,hotpotqa,What year did Guns N Roses perform a promo for a movie starring Arnold Schwarzenegger as a former New York Police detective?,1999,"[ollows former New York Police Department detective Jericho Cane (Schwarzenegger) after he saves a banker (Byrne) from an assassin, finds himself embroiled in a religious conflict, and must protec...",bm25,3,retrieval_results_bm25_3_13.json,instruction_tuned,Qwen/Qwen2.5-7B-Instruct,instruct,1999


## Combine and score results


In [18]:

comparison_all_results = pd.concat([comparison_base_results, comparison_instruct_results], ignore_index=True)

comparison_all_results["exact_match"] = comparison_all_results.apply(
    lambda row: exact_match_score(row["prediction"], row["answer"]),
    axis=1,
)
comparison_all_results["token_f1"] = comparison_all_results.apply(
    lambda row: token_f1_score(row["prediction"], row["answer"]),
    axis=1,
)
comparison_all_results["abstained"] = all_results["prediction"].apply(is_abstention).astype(float)
comparison_all_results["prediction_in_context"] = comparison_all_results.apply(
    lambda row: prediction_in_context(row["prediction"], row["contexts"]),
    axis=1,
)

comparison_summary_by_domain = (
    comparison_all_results
    .groupby(["domain", "model_label"], as_index=False)
    .agg(
        n=("id", "count"),
        em=("exact_match", "mean"),
        f1=("token_f1", "mean"),
        abstain_rate=("abstained", "mean"),
        pred_in_context_rate=("prediction_in_context", "mean"),
    )
    .sort_values(["domain", "model_label"])
)

comparison_summary_overall = (
    comparison_all_results
    .groupby(["model_label"], as_index=False)
    .agg(
        n=("id", "count"),
        em=("exact_match", "mean"),
        f1=("token_f1", "mean"),
        abstain_rate=("abstained", "mean"),
        pred_in_context_rate=("prediction_in_context", "mean"),
    )
    .sort_values(["model_label"])
)

display(comparison_summary_by_domain)
display(comparison_summary_overall)


,domain,model_label,n,em,f1,abstain_rate,pred_in_context_rate
0,hotpotqa,base_zero_shot,8,0.25,0.285714,0.625,0.375
1,hotpotqa,instruction_tuned,8,0.50,0.598214,0.250,0.625


,model_label,n,em,f1,abstain_rate,pred_in_context_rate
0,base_zero_shot,8,0.25,0.285714,0.625,0.375
1,instruction_tuned,8,0.50,0.598214,0.250,0.625


## Save outputs


In [19]:
comparison_raw_csv_path = OUTPUT_DIR / f"{COMPARISON_OUTPUT_PREFIX}_raw_results.csv"
comparison_domain_csv_path = OUTPUT_DIR / f"{COMPARISON_OUTPUT_PREFIX}_summary_by_domain.csv"
comparison_overall_csv_path = OUTPUT_DIR / f"{COMPARISON_OUTPUT_PREFIX}_summary_overall.csv"
comparison_jsonl_path = OUTPUT_DIR / f"{COMPARISON_OUTPUT_PREFIX}_raw_results.jsonl"

comparison_all_results.to_csv(comparison_raw_csv_path, index=False)
comparison_summary_by_domain.to_csv(comparison_domain_csv_path, index=False)
comparison_summary_overall.to_csv(comparison_overall_csv_path, index=False)

with open(comparison_jsonl_path, "w", encoding="utf-8") as f:
    for row in comparison_all_results.to_dict("records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:")
print("-", comparison_raw_csv_path)
print("-", comparison_domain_csv_path)
print("-", comparison_overall_csv_path)
print("-", comparison_jsonl_path)


Saved:
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_raw_results.csv
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_summary_by_domain.csv
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_summary_overall.csv
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_raw_results.jsonl


## Quick qualitative comparison

This cell shows a few side-by-side examples so you can inspect whether the instruction-tuned model is more stable, grounded, or concise.


In [20]:

wide_compare = (
    comparison_all_results[["id", "domain", "question", "answer", "model_label", "prediction"]]
    .pivot_table(
        index=["id", "domain", "question", "answer"],
        columns="model_label",
        values="prediction",
        aggfunc="first",
    )
    .reset_index()
)

display(wide_compare.head(10))


model_label,id,domain,question,answer,base_zero_shot,instruction_tuned
0,0,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,Insufficient evidence.,Yes.
1,11,hotpotqa,"What screenwriter with credits for ""Evolution"" co-wrote a film starring Nicolas Cage and Téa Leoni?",David Weissman,Insufficient evidence.,David Diamond
2,12,hotpotqa,What year did Guns N Roses perform a promo for a movie starring Arnold Schwarzenegger as a former New York Police detective?,1999,1999,1999
3,14,hotpotqa,The football manager who recruited David Beckham managed Manchester United during what timeframe?,from 1986 to 2013,1945 to 1969,1945 to 1969
4,15,hotpotqa,Brown State Fishing Lake is in a country that has a population of how many inhabitants ?,"9,984",Insufficient evidence.,Insufficient evidence.
5,16,hotpotqa,The Vermont Catamounts men's soccer team currently competes in a conference that was formerly known as what from 1988 to 1996?,the North Atlantic Conference,North Atlantic Conference,North Atlantic Conference
6,19,hotpotqa,"Which writer was from England, Henry Roth or Robert Erskine Childers?",Robert Erskine Childers DSC,Insufficient evidence.,Insufficient evidence.
7,3,hotpotqa,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,Insufficient evidence.,No.


## Optional: export a template JSONL file for your teammate

This is useful if you want to give your retrieval teammate a concrete schema to follow.


In [21]:

template_records = eval_df.head(2).to_dict("records")
template_path = OUTPUT_DIR / "retrieval_jsonl_template.jsonl"

with open(template_path, "w", encoding="utf-8") as f:
    for obj in template_records:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("Template saved to:", template_path)


Template saved to: /content/cs6493_topic4_generation/outputs/retrieval_jsonl_template.jsonl


## Optional: teammate repo reference

Your teammate's repo is:

`https://github.com/CS6493/RAGProjectPreprocess`

You do **not** need that repo for this notebook's **oracle mode**.

But if you want to keep everything in one Colab workspace, you can clone it here for reference:

```python
!git clone https://github.com/CS6493/RAGProjectPreprocess.git
```

For your final report, the best workflow is usually:

1. teammate runs retrieval and exports JSONL  
2. you switch this notebook to `INPUT_MODE = "team_jsonl"`  
3. you run the two generation models and compare outputs


## Troubleshooting

### 1) CUDA out of memory
Try one or more of these:
- switch `MODEL_PRESET` from `qwen_7b` to `qwen_3b`
- reduce `MAX_NEW_TOKENS`
- reduce `SAMPLES_PER_DOMAIN`
- restart runtime and run again
- keep `USE_4BIT = True`

### 2) Slow first run
The first run downloads models and datasets. Later runs are faster if you cache them in Google Drive.

### 3) Why oracle mode is not enough for the final report
Oracle mode is mainly for:
- debugging the generation module
- comparing base vs instruct behavior
- getting a self-contained Colab pipeline working quickly

But it does **not** properly test retrieval quality differences.  
For the real Topic 4 retrieval–generation analysis, use **team JSONL mode** with actual retrieval outputs.


# 检查PubMedQA的输入

In [22]:
from datasets import load_dataset

pubmed_debug_ds = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train"
)

sample = pubmed_debug_ds[0]

print(type(sample))
print(sample.keys())
print(type(sample["context"]))
print(sample["context"])
print(sample["final_decision"])

<class 'dict'>
dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision'])
<class 'dict'>
{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), 

# 修改pubMedQA并检查

In [23]:
test_item = pubmed_debug_ds[0]
test_record = pubmedqa_to_record(test_item)

print(test_record["question"])
print(test_record["answer"])
print(len(test_record["contexts"]))
print(test_record["contexts"][0][:800])

Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
yes
1
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; ce


In [24]:
pubmed_rows = eval_df[eval_df["domain"] == "pubmedqa"]
print(pubmed_rows[["question", "answer", "contexts"]].head(3))

Empty DataFrame
Columns: [question, answer, contexts]
Index: []


In [25]:
# print(len(pubmed_rows))
# print(pubmed_rows.iloc[0]["question"])
# print(pubmed_rows.iloc[0]["answer"])
# print(pubmed_rows.iloc[0]["contexts"])

# 检查BM25对齐数量

In [26]:
import json
from pathlib import Path

bm25_files = sorted(INPUT_DIR.glob("retrieval_results_bm25_3_*.json"))

print("number of bm25 files:", len(bm25_files))
print()

bad_files = []
ok_files = []

for fp in bm25_files:
    with open(fp, "r", encoding="utf-8") as f:
        obj = json.load(f)

    query = str(obj.get("query", "")).strip()
    results = obj.get("results", [])

    meta_questions = [str(r.get("metadata", {}).get("question", "")).strip() for r in results]
    meta_answers = [str(r.get("metadata", {}).get("answer", "")).strip() for r in results]

    unique_meta_questions = set(meta_questions)
    unique_meta_answers = set(meta_answers)

    is_bad = False
    reason = []

    if len(unique_meta_questions) != 1:
        is_bad = True
        reason.append("metadata.question not unique")

    if len(unique_meta_answers) != 1:
        is_bad = True
        reason.append("metadata.answer not unique")

    if len(unique_meta_questions) == 1:
        only_meta_question = list(unique_meta_questions)[0]
        if query != only_meta_question:
            is_bad = True
            reason.append("query != metadata.question")

    if is_bad:
        bad_files.append({
            "file": fp.name,
            "query": query,
            "meta_questions": meta_questions,
            "meta_answers": meta_answers,
            "reason": reason,
        })
    else:
        ok_files.append(fp.name)

print("OK files:", len(ok_files))
print("BAD files:", len(bad_files))
print()

if bad_files:
    print("Examples of BAD files:")
    print("=" * 100)
    for item in bad_files[:10]:
        print("file:", item["file"])
        print("reason:", item["reason"])
        print("query:", item["query"])
        print("metadata.questions:")
        for q in item["meta_questions"]:
            print(" -", q)
        print("metadata.answers:")
        for a in item["meta_answers"]:
            print(" -", a)
        print("=" * 100)
else:
    print("All checked BM25 files look aligned.")

number of bm25 files: 20

OK files: 8
BAD files: 12

Examples of BAD files:
file: retrieval_results_bm25_3_10.json
reason: ['metadata.question not unique', 'metadata.answer not unique']
query: Are Local H and For Against both from the United States?
metadata.questions:
 - What is the mascot of the oldest private university in Georgia?
 - Are both American bands The Four Seasons and Local H popular around the Beatles were around?
 - Kathleen Matthews holds what political position in the Maryland Democratic Party?
metadata.answers:
 - The Bears
 - no
 - current state party chair
file: retrieval_results_bm25_3_11.json
reason: ['metadata.question not unique', 'metadata.answer not unique']
query: What is the name of the fight song of the university whose main campus is in Lawrence, Kansas and whose branch campuses are in the Kansas City metropolitan area?
metadata.questions:
 - What is the name of the fight song of the university whose main campus is in Lawrence, Kansas and whose branch cam